In [11]:
import pymupdf

pdf_configs = [
    {"path": "./resources/Hands-On Large Language Models.pdf", "start": 23, "end": 411},
    {"path": "./resources/Introducing-MLOps.pdf", "start": 15, "end": 171},
]

raw_text = ""
for cfg in pdf_configs:
    doc = pymupdf.open(cfg["path"])
    for page in doc:
        if page.number < cfg["start"] or page.number > cfg["end"]:
            continue

        blocks = page.get_text("blocks")
        for block in blocks:
            y1 = block[3]
            page_height = page.rect.height
            if y1 < page_height * 0.92:
                raw_text += block[4]

    doc.close()

print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:100], end="\n\n")

print(raw_text[-100:])

Total number of characters: 877108
CHAPTER 1
An Introduction to Large Language Models
Humanity is at an inflection point. From 2012 onw

llows data teams (and more widely, entire organiza‐
tions) to scale their machine learning efforts.



In [ ]:
import re

# words = re.split(r'(\s)', raw_text)
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [word for word in preprocessed if word.strip()]

print(f"Total length of preprocessed wor: {len(preprocessed)}")

all_tokens = sorted(list(set(preprocessed)))
vocab_size = len(all_tokens)

print(f"Vocab length of dataset: {vocab_size}")

Total length of preprocessed wor: 165539
Vocab length of dataset: 12056


212121

In [17]:
import torch
from torch.utils.data import DataLoader, Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        print(f"length of token_ids: {len(token_ids)}")

        for idx in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[idx: idx + max_length]
            target_chunk = token_ids[idx + 1: idx + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [18]:
import tiktoken

def create_dataloader_v1(
    txt,           # raw text string to tokenize and chunk
    batch_size=4,  # number of samples per batch
    max_length=256,# number of tokens in each input/target chunk (context window)
    stride=128,    # how many tokens to slide the window forward; overlap when stride < max_length
    shuffle=True,  # shuffle batches each epoch (disable for evaluation)
    drop_last=True,# drop the final incomplete batch so all batches are equal size
    num_workers=10 # parallel workers for data loading
):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [19]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False,
    num_workers=0
)

data_iter = iter(dataloader)
print(data_iter)

length of token_ids: 212121


In [5]:
next_batch = next(data_iter)
print(next_batch)

[tensor([[41481,   352,   198,  2025],
        [22395,   284, 13601, 15417],
        [32329,   198, 20490,   414],
        [  318,   379,   281,  1167],
        [ 1564,   966,    13,  3574],
        [ 2321, 31719,    11, 13312],
        [  287,  2615,  9552,   198],
        [10057,    82,   357,  3500]]), tensor([[  352,   198,  2025, 22395],
        [  284, 13601, 15417, 32329],
        [  198, 20490,   414,   318],
        [  379,   281,  1167,  1564],
        [  966,    13,  3574,  2321],
        [31719,    11, 13312,   287],
        [ 2615,  9552,   198, 10057],
        [   82,   357,  3500,  2769]])]
